In [1]:
# 0. 라이브러리 임포트
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print('✅ 라이브러리 로드 완료')

✅ 라이브러리 로드 완료


In [2]:
# 1. 데이터 로드
df = pd.read_excel('../data/Ecommerce_Dataset.xlsx', sheet_name='E Comm')

print(f'📊 Data shape: {df.shape[0]:,} rows x {df.shape[1]} cols')
df.head()

📊 Data shape: 5,630 rows x 20 cols


,CustomerID,Churn,Tenure,PreferredLoginDevice,CityTier,WarehouseToHome,PreferredPaymentMode,Gender,HourSpendOnApp,NumberOfDeviceRegistered,PreferedOrderCat,SatisfactionScore,MaritalStatus,NumberOfAddress,Complain,OrderAmountHikeFromlastYear,CouponUsed,OrderCount,DaySinceLastOrder,CashbackAmount
0,50001,1,4.0,Mobile Phone,3,6.0,Debit Card,Female,3.0,3,Laptop & Accessory,2,Single,9,1,11.0,1.0,1.0,5.0,159.93
1,50002,1,NaN,Phone,1,8.0,UPI,Male,3.0,4,Mobile,3,Single,7,1,15.0,0.0,1.0,0.0,120.90
2,50003,1,NaN,Phone,1,30.0,Debit Card,Male,2.0,4,Mobile,3,Single,6,1,14.0,0.0,1.0,3.0,120.28
3,50004,1,0.0,Phone,3,15.0,Debit Card,Male,2.0,4,Laptop & Accessory,5,Single,8,0,23.0,0.0,1.0,3.0,134.07
4,50005,1,0.0,Phone,1,12.0,CC,Male,NaN,3,Mobile,5,Single,3,0,11.0,1.0,1.0,3.0,129.60


In [3]:
# 2. 중복 제거
before = df.shape[0]
df = df.drop_duplicates()
print(f'Duplicates removed: {before - df.shape[0]}')
print(f'After: {df.shape[0]:,} rows')

Duplicates removed: 0
After: 5,630 rows


In [4]:
# 3. 결측치 확인
print('=== Missing Values ===')
missing = pd.DataFrame({
    'null_count': df.isnull().sum(),
    'null_%': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing[missing['null_count'] > 0])

=== Missing Values ===
                             null_count  null_%
Tenure                              264    4.69
WarehouseToHome                     251    4.46
HourSpendOnApp                      255    4.53
OrderAmountHikeFromlastYear         265    4.71
CouponUsed                          256    4.55
OrderCount                          258    4.58
DaySinceLastOrder                   307    5.45


In [5]:
# 4. 결측치 대체
# 논문 기준:
# - 범위 넓고 다른 변수와 상관있는 컬럼 → Iterative Imputer
# - HourSpendOnApp → Median

iterative_cols = [
    'DaySinceLastOrder', 'OrderAmountHikeFromlastYear',
    'Tenure', 'OrderCount', 'CouponUsed', 'WarehouseToHome'
]
median_cols = ['HourSpendOnApp']

# Iterative Imputer
imp = IterativeImputer(max_iter=10, random_state=42)
df[iterative_cols] = imp.fit_transform(df[iterative_cols])

# Median
for col in median_cols:
    df[col] = df[col].fillna(df[col].median())

print('✅ Missing values imputed')
print(f'Remaining missing: {df.isnull().sum().sum()}')

✅ Missing values imputed
Remaining missing: 0


In [6]:
# 5. One-Hot Encoding
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Categorical columns: {cat_cols}')

df = pd.get_dummies(df, columns=cat_cols, drop_first=False)
print(f'After encoding: {df.shape[1]} cols')

Categorical columns: ['PreferredLoginDevice', 'PreferredPaymentMode', 'Gender', 'PreferedOrderCat', 'MaritalStatus']
After encoding: 36 cols


In [7]:
# 6. 이상치 제거 (Mahalanobis Distance, p < 0.001)
# 논문: 266개 제거 → 최종 4,807개

num_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
X_num = df[num_cols].values

cov = np.cov(X_num.T)
inv_cov = np.linalg.pinv(cov)
mean = np.mean(X_num, axis=0)

diff = X_num - mean
mahal = np.sqrt(np.einsum('ij,jk,ik->i', diff, inv_cov, diff))
p_values = 1 - stats.chi2.cdf(mahal**2, df=X_num.shape[1])

outlier_mask = p_values >= 0.001
before = df.shape[0]
df = df[outlier_mask].reset_index(drop=True)

# 논문 기준: 266개 제거 → 4,807개
# 재현 결과: 102개 제거 → 5,528개
# (데이터 버전 및 imputer 시드 차이로 인한 불일치)
print(f'Outliers removed: {before - df.shape[0]}')
print(f'Final data: {df.shape[0]:,} rows')  

Outliers removed: 102
Final data: 5,528 rows


In [8]:
# 7. Train/Test Split (80:20, stratified)
from sklearn.model_selection import train_test_split

X = df.drop(columns=['Churn'])
y = df['Churn']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train : {X_train.shape[0]:,} rows ({y_train.mean()*100:.1f}% churn)')
print(f'Test  : {X_test.shape[0]:,} rows ({y_test.mean()*100:.1f}% churn)')

Train : 4,422 rows (16.8% churn)
Test  : 1,106 rows (16.8% churn)


In [9]:
# 8. 전처리 데이터 저장 (다음 노트북에서 사용)
import os
os.makedirs('../outputs', exist_ok=True)

X_train.to_csv('../outputs/X_train.csv', index=False)
X_test.to_csv('../outputs/X_test.csv', index=False)
y_train.to_csv('../outputs/y_train.csv', index=False)
y_test.to_csv('../outputs/y_test.csv', index=False)

print('✅ Preprocessed data saved to outputs/')
print('➡️  Next: 03_modeling.ipynb')

✅ Preprocessed data saved to outputs/
➡️  Next: 03_modeling.ipynb
